# RAG Assignment — Bi-Encoder Baseline vs. Cross-Encoder Re-Ranking

**Fresh run from extracted text files only.** No previous chunks, embeddings, rankings, cached answers, or original PDF are required.

The notebook loads extracted `.txt` files, chunks by page, embeds and retrieves Top 10, reranks those same 10 candidates, compares the baseline Top 3 against the reranked Top 3, generates paired answers with the same local LLM prompt/settings, profiles latency, exports review tables, and creates a ZIP of all outputs.

**Before running:** attach the extracted text-file folder as a Kaggle input or keep it near this notebook in VS Code. Set `CORPUS_PATH` in Section 1 if automatic discovery does not find the right folder. GPU recommended. The default generator uses public Qwen Instruct models (with a smaller-model fallback) rather than Gemma 3, because the earlier Gemma run produced NaN logits.

The assignment asks for at least two **genuine** failure cases. Promotion in rank alone is not evidence of success: manually verify candidate passages and answer comparisons before claiming reranking worked.

## 0. Install dependencies

Run once. Internet access is needed to download models from Hugging Face.

In [1]:
%pip install "sentence-transformers>=3.4.0,<6.0" "transformers>=4.51.0,<5.0" "accelerate>=1.0.0" "bitsandbytes>=0.45.0" tqdm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 92.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 49.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 24.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 81.9 MB/s eta 0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.29.0
    Uninstalling huggingface_hub-1.29.0:
      Successfully uninstalled huggingface_hub-1.29.0
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.23.1
    Uninstalling tokenizers-0.23.1:
      Successfully uninstalled tokenizers-0.23.1
  Attempting uninstall: transformers
    Found existing installation: transformers 5.16.1
    Uninstalling transformers-5.16.1:
      Successfully uninstalled transformers-5.16.1
ERROR: pip's dependency resolver does not currently take into a

## 1. Configuration and output setup

`CORPUS_PATH` may be a directory of page-level `.txt` files or one full-text `.txt` file. Leave it as `None` to auto-detect common Kaggle/local locations.

In [2]:
from pathlib import Path
import os, re, json, time, gc, shutil, hashlib, platform, warnings, zipfile
from collections import defaultdict

import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm

# ----------------------------- CONFIG ---------------------------------
CORPUS_PATH = None  # e.g. "/kaggle/input/adobe-extracted" or "./adobe_10k_extracted"
CHUNK_SIZE_WORDS = 350
CHUNK_OVERLAP_WORDS = 60
MIN_CHUNK_WORDS = 40
TOP_K_CANDIDATES = 10
TOP_K_CONTEXT = 3

EMBEDDING_MODEL_NAME = "BAAI/bge-base-en-v1.5"
RERANKER_MODEL_NAME = "BAAI/bge-reranker-base"
GENERATION_MODEL_CANDIDATES = [
    "Qwen/Qwen2.5-3B-Instruct",
    "Qwen/Qwen2.5-1.5B-Instruct",
]
GENERATION_MAX_NEW_TOKENS = 1024
USE_4BIT_GENERATOR_WHEN_GPU = True

IN_KAGGLE = Path("/kaggle/working").exists() and Path("/kaggle/input").exists()
BASE_DIR = Path("/kaggle/working") if IN_KAGGLE else Path.cwd()
OUTPUT_DIR = BASE_DIR / "rag_outputs"
ZIP_PATH = BASE_DIR / "rag_assignment_outputs.zip"
DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"

# Fresh run: clear only this notebook's result directory.
if OUTPUT_DIR.exists():
    shutil.rmtree(OUTPUT_DIR)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

np.random.seed(42)
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

print("Environment:", "Kaggle" if IN_KAGGLE else "Local / VS Code")
print("Python:", platform.python_version(), "| PyTorch:", torch.__version__)
print("Device:", DEVICE)
print("Output directory:", OUTPUT_DIR.resolve())
print("ZIP destination:", ZIP_PATH.resolve())
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        free, total = torch.cuda.mem_get_info(i)
        print(f"GPU {i}: {torch.cuda.get_device_name(i)} | {free/1024**3:.2f}/{total/1024**3:.2f} GiB free")

Environment: Kaggle
Python: 3.13.15 | PyTorch: 2.11.0+cu128
Device: cuda:0
Output directory: /kaggle/working/rag_outputs
ZIP destination: /kaggle/working/rag_assignment_outputs.zip
GPU 0: Tesla T4 | 14.46/14.56 GiB free
GPU 1: Tesla T4 | 14.46/14.56 GiB free


## 2. Find and load extracted `.txt` files

Page files such as `page_1.txt`, `page_002.txt`, `ocr_page_3.txt`, and `001.txt` are ordered by page number. A single text file is also supported: form-feed or explicit `Page N` markers are used when present. If there are no page markers in a single file, page citations can only refer to page 1.

In [3]:
def page_number_from_stem(path):
    stem = path.stem.lower()
    for pattern in (r"(?:ocr[_-]?)?page[_-]?(\d+)$", r"^p[_-]?(\d+)$", r"^(\d{1,5})$"):
        match = re.match(pattern, stem)
        if match:
            return int(match.group(1))
    return None

def clean_text(text):
    text = text.replace("\x00", " ").replace("\r\n", "\n").replace("\r", "\n")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()

def discover_page_folder(root):
    groups = defaultdict(list)
    if not root.exists() or not root.is_dir():
        return None, []
    try:
        for path in root.rglob("*.txt"):
            if any(part.startswith(".") for part in path.parts):
                continue
            page_no = page_number_from_stem(path)
            if page_no is not None and path.is_file():
                groups[path.parent].append((page_no, path))
    except (PermissionError, OSError):
        pass
    if not groups:
        return None, []
    folder, records = max(groups.items(), key=lambda x: len(x[1]))
    records.sort(key=lambda x: (x[0], x[1].name.lower()))
    return folder, records

def discover_corpus_path():
    roots = []
    if IN_KAGGLE:
        roots.extend([
            Path("/kaggle/input/datasets/vidurshah04/adobe-extracted"),
            Path("/kaggle/input/adobe-extracted"),
            Path("/kaggle/input"),
        ])
    roots.extend([Path.cwd() / "adobe_10k_extracted", Path.cwd() / "extracted_text", Path.cwd()])
    seen = set()
    for root in roots:
        try:
            root = root.resolve()
        except OSError:
            continue
        if str(root) in seen or not root.exists():
            continue
        seen.add(str(root))
        if root.is_file() and root.suffix.lower() == ".txt" and root.stat().st_size > 1000:
            return root
        folder, records = discover_page_folder(root)
        if folder is not None and len(records) >= 2:
            return folder
    # As a last resort, discover a named, sizeable full-text file.
    for root in roots:
        if not root.exists() or not root.is_dir():
            continue
        try:
            files = [p for p in root.rglob("*.txt") if p.is_file() and p.stat().st_size > 10_000
                     and re.search(r"(adobe|10k|10-k|extracted|full.?text)", p.stem, re.I)]
        except (PermissionError, OSError):
            files = []
        if files:
            return max(files, key=lambda p: p.stat().st_size)
    return None

def split_single_text(text):
    text = text.replace("\r\n", "\n").replace("\r", "\n")
    if "\f" in text:
        parts = [clean_text(p) for p in text.split("\f")]
        parts = [p for p in parts if p]
        if len(parts) > 1:
            return [{"page": i, "text": part} for i, part in enumerate(parts, 1)]
    marker = re.compile(r"(?im)^\s*(?:={2,}\s*)?(?:page|pdf page)\s+(\d+)(?:\s*={2,})?\s*$")
    matches = list(marker.finditer(text))
    if len(matches) >= 2:
        pages = []
        for i, m in enumerate(matches):
            end = matches[i + 1].start() if i + 1 < len(matches) else len(text)
            part = clean_text(text[m.end():end])
            if part:
                pages.append({"page": int(m.group(1)), "text": part})
        if pages:
            return pages
    return [{"page": 1, "text": clean_text(text)}]

def load_pages(path_value=None):
    candidate = Path(path_value).expanduser() if path_value else discover_corpus_path()
    if candidate is None:
        raise FileNotFoundError(
            "Could not auto-detect extracted text. Set CORPUS_PATH to the exact text folder or full .txt file."
        )
    candidate = candidate.resolve()
    if candidate.is_file():
        if candidate.suffix.lower() != ".txt":
            raise ValueError(f"Expected .txt input, got {candidate}")
        pages = split_single_text(candidate.read_text(encoding="utf-8", errors="replace"))
        source = str(candidate)
    elif candidate.is_dir():
        folder, record_files = discover_page_folder(candidate)
        if record_files:
            pages = [{"page": number, "text": clean_text(file.read_text(encoding="utf-8", errors="replace"))}
                      for number, file in record_files]
            source = str(folder)
        else:
            text_files = sorted([p for p in candidate.glob("*.txt") if p.is_file()], key=lambda p: p.name.lower())
            if len(text_files) >= 2:
                pages = [{"page": i, "text": clean_text(p.read_text(encoding="utf-8", errors="replace"))}
                          for i, p in enumerate(text_files, 1)]
                source = str(candidate)
            elif len(text_files) == 1 and text_files[0].stat().st_size > 1000:
                pages = split_single_text(text_files[0].read_text(encoding="utf-8", errors="replace"))
                source = str(text_files[0])
            else:
                raise FileNotFoundError(f"No usable extracted .txt files found in {candidate}")
    else:
        raise FileNotFoundError(f"Input path does not exist: {candidate}")
    pages = [{"page": int(p["page"]), "text": clean_text(p["text"])} for p in pages if clean_text(p["text"])]
    if not pages:
        raise ValueError("The text input was empty after cleaning.")
    return source, pages

corpus_source, pages = load_pages(CORPUS_PATH)
print("Loaded from:", corpus_source)
print("Text records/pages:", len(pages))
print("Approximate words:", sum(len(p["text"].split()) for p in pages))
print("First record page:", pages[0]["page"])
print(pages[0]["text"][:500])
if len(pages) == 1 and pages[0]["page"] == 1:
    print("Note: page boundaries were not detectable; citations will refer to p. 1.")

Loaded from: /kaggle/input/datasets/vidurshah04/adobe-extracted
Text records/pages: 93
Approximate words: 51201
First record page: 1
UNITED STATES

SECURITIES AND EXCHANGE COMMISSION
Washington, D.C. 20549

FORM 10-K

(Mark One)
x} ANNUAL REPORT PURSUANT TO SECTION 13 OR 15(d) OF THE SECURITIES EXCHANGE ACT OF 1934
For the fiscal year ended November 28, 2025
or
TRANSITION REPORT PURSUANT TO SECTION 13 OR 15(d) OF THE SECURITIES EXCHANGE ACT OF 1934

For the transition period from to
Commission File Number: 000-15175

ADOBE INC.

(Exact name of registrant as specified in its charter)

Delaware 77-0019522
(State or other jurisd


## 3. Chunk each page and save normalized text

Page boundaries are retained in the metadata. IDs are assigned in reading order, giving each chunk a stable ID for rank-shift comparisons.

In [4]:
def build_chunks(page_records):
    chunks = []
    if CHUNK_OVERLAP_WORDS >= CHUNK_SIZE_WORDS:
        raise ValueError("Overlap must be smaller than chunk size.")
    for page in page_records:
        words = page["text"].split()
        start = 0
        while start < len(words):
            end = min(start + CHUNK_SIZE_WORDS, len(words))
            text = " ".join(words[start:end]).strip()
            if len(text.split()) >= MIN_CHUNK_WORDS:
                chunks.append({
                    "chunk_id": len(chunks),
                    "page": int(page["page"]),
                    "word_count": len(text.split()),
                    "text": text,
                })
            if end >= len(words):
                break
            start = end - CHUNK_OVERLAP_WORDS
    return chunks

chunks = build_chunks(pages)
if not chunks:
    raise ValueError("No chunks were made. Check the extracted text and chunk size.")
chunk_texts = [c["text"] for c in chunks]
(OUTPUT_DIR / "normalized_chunks.json").write_text(
    json.dumps(chunks, ensure_ascii=False, indent=2), encoding="utf-8"
)
(OUTPUT_DIR / "corpus_source.txt").write_text(
    f"Source: {corpus_source}\nText records/pages loaded: {len(pages)}\nChunks: {len(chunks)}\n",
    encoding="utf-8",
)
print("Chunks:", len(chunks), "| unique pages:", len(set(c["page"] for c in chunks)))
print("Average words/chunk:", round(np.mean([c["word_count"] for c in chunks]), 1))
display(pd.DataFrame(chunks[:3])[["chunk_id", "page", "word_count", "text"]])

Chunks: 207 | unique pages: 93
Average words/chunk: 280.4


,chunk_id,page,word_count,text
0,0,1,350,UNITED STATES SECURITIES AND EXCHANGE COMMISSI...
1,1,1,350,of this chapter) during the preceding 12 month...
2,2,1,154,"$0.0001 par value per share, held by non-affil..."


## 4. Build the bi-encoder index and cross-encoder

The BGE bi-encoder creates normalized vectors for every chunk. Similarity is calculated by vector dot product, equivalent to cosine similarity for normalized vectors. No previous embeddings are read or reused.

In [5]:
from sentence_transformers import SentenceTransformer, CrossEncoder

def sync_cuda():
    if torch.cuda.is_available():
        torch.cuda.synchronize()

QUERY_PREFIX = "Represent this sentence for searching relevant passages: " if "bge" in EMBEDDING_MODEL_NAME.lower() else ""

t0 = time.perf_counter()
embedder = SentenceTransformer(EMBEDDING_MODEL_NAME, device=DEVICE)
bi_encoder_load_s = time.perf_counter() - t0

t0 = time.perf_counter()
embeddings = embedder.encode(
    chunk_texts, batch_size=32, show_progress_bar=True,
    convert_to_numpy=True, normalize_embeddings=True,
).astype(np.float32)
sync_cuda()
embedding_time_s = time.perf_counter() - t0

if embeddings.shape[0] != len(chunks):
    raise ValueError("Embedding row count does not match chunk count.")
if not np.isfinite(embeddings).all():
    raise ValueError("Embedding matrix contains NaN/Inf.")
np.save(OUTPUT_DIR / "embeddings.npy", embeddings)

t0 = time.perf_counter()
reranker = CrossEncoder(RERANKER_MODEL_NAME, device=DEVICE)
cross_encoder_load_s = time.perf_counter() - t0

model_load_df = pd.DataFrame([
    {"component": "bi_encoder_model_load", "seconds": bi_encoder_load_s},
    {"component": "corpus_embedding", "seconds": embedding_time_s},
    {"component": "cross_encoder_model_load", "seconds": cross_encoder_load_s},
])
model_load_df.to_csv(OUTPUT_DIR / "model_load_profile.csv", index=False)
print("Embedding matrix:", embeddings.shape)
print(model_load_df)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/777 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/443 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/279 [00:00<?, ?B/s]

Embedding matrix: (207, 768)
                  component   seconds
0     bi_encoder_model_load  5.506104
1          corpus_embedding  6.038167
2  cross_encoder_model_load  9.821676


## 5. Evaluation queries

The set includes broad questions and document-specific stress-test candidates for cybersecurity details, legal proceedings, and indemnification qualifications. These are hypotheses for stress-testing, not pre-declared reranking successes.

In [6]:
QUERY_SPECS = [
    {
        "query": "How does Adobe incorporate generative AI into its products and services, and what related risks does it disclose?",
        "query_type": "broad coverage",
        "failure_mode_candidate": "Terminology / risk-context distractor",
    },
    {
        "query": "What cybersecurity risks does Adobe disclose, and how does the company manage cybersecurity through oversight, assessment, and response practices?",
        "query_type": "stress-test candidate",
        "failure_mode_candidate": "Keyword trap / governance details",
    },
    {
        "query": "What were Adobe's Digital Media and Digital Experience subscription revenues in fiscal 2025 compared with fiscal 2024?",
        "query_type": "numeric disclosure",
        "failure_mode_candidate": "Similar segment names / numeric distractor",
    },
    {
        "query": "What major factors could adversely affect Adobe's business, financial condition, or results of operations?",
        "query_type": "broad coverage",
        "failure_mode_candidate": "General risk disclosure",
    },
    {
        "query": "What changes does Adobe report in its financial performance for fiscal 2025?",
        "query_type": "broad coverage",
        "failure_mode_candidate": "Financial results / similar figures",
    },
    {
        "query": "What legal proceedings does Adobe disclose, including the status of securities class actions and related derivative actions?",
        "query_type": "stress-test candidate",
        "failure_mode_candidate": "Specific legal detail vs general litigation",
    },
    {
        "query": "What qualifications or limitations apply to Adobe's indemnification obligations for third-party intellectual-property infringement claims?",
        "query_type": "stress-test candidate",
        "failure_mode_candidate": "Exception / limitation",
    },
    {
        "query": "What were subscription revenues from Creative and Marketing Professionals compared with Business Professionals and Consumers in fiscal 2025 and fiscal 2024?",
        "query_type": "numeric disclosure",
        "failure_mode_candidate": "Similar customer-group labels / numeric distractor",
    },
    {
        "query": "Which cybersecurity oversight, risk assessment, and incident-response processes does Adobe disclose, apart from general market and financial risks?",
        "query_type": "stress-test candidate",
        "failure_mode_candidate": "Keyword trap / distinct terminology",
    },
    {
        "query": "What does Adobe disclose about indemnifying customers or channel partners for third-party intellectual-property claims, including qualifications to the obligation?",
        "query_type": "stress-test candidate",
        "failure_mode_candidate": "Exception / role-specific constraint",
    },
]
ALL_QUERIES = [spec["query"] for spec in QUERY_SPECS]
query_metadata = {s["query"]: s for s in QUERY_SPECS}
pd.DataFrame([{"query_id": i, **q} for i, q in enumerate(QUERY_SPECS, 1)]).to_csv(
    OUTPUT_DIR / "evaluation_queries.csv", index=False
)
print("Queries:", len(ALL_QUERIES), "| target generated answers:", 2 * len(ALL_QUERIES))
display(pd.DataFrame(QUERY_SPECS))

Queries: 10 | target generated answers: 20


,query,query_type,failure_mode_candidate
0,How does Adobe incorporate generative AI into ...,broad coverage,Terminology / risk-context distractor
1,"What cybersecurity risks does Adobe disclose, ...",stress-test candidate,Keyword trap / governance details
2,What were Adobe's Digital Media and Digital Ex...,numeric disclosure,Similar segment names / numeric distractor
3,What major factors could adversely affect Adob...,broad coverage,General risk disclosure
4,What changes does Adobe report in its financia...,broad coverage,Financial results / similar figures
5,"What legal proceedings does Adobe disclose, in...",stress-test candidate,Specific legal detail vs general litigation
6,What qualifications or limitations apply to Ad...,stress-test candidate,Exception / limitation
7,What were subscription revenues from Creative ...,numeric disclosure,Similar customer-group labels / numeric distra...
8,"Which cybersecurity oversight, risk assessment...",stress-test candidate,Keyword trap / distinct terminology
9,What does Adobe disclose about indemnifying cu...,stress-test candidate,Exception / role-specific constraint


## 6. Retrieve Top 10 once, then rerank those same candidates

For every question, the dense bi-encoder retrieves Top 10. The cross-encoder scores only these ten query–chunk pairs and sorts them by its own relevance score. The two contexts sent to the LLM are dense Top 3 and reranked Top 3.

In [7]:
def dense_retrieve(query, k=TOP_K_CANDIDATES):
    sync_cuda()
    start = time.perf_counter()
    qvec = embedder.encode(
        [QUERY_PREFIX + query], convert_to_numpy=True,
        normalize_embeddings=True, show_progress_bar=False,
    )[0].astype(np.float32)
    scores = embeddings @ qvec
    ids = np.argsort(scores)[::-1][:min(k, len(chunks))]
    sync_cuda()
    elapsed_ms = (time.perf_counter() - start) * 1000.0
    candidates = []
    for rank, idx in enumerate(ids, 1):
        c = chunks[int(idx)]
        candidates.append({
            "chunk_id": int(c["chunk_id"]),
            "page": int(c["page"]),
            "text": c["text"],
            "dense_rank": int(rank),
            "dense_score": float(scores[int(idx)]),
        })
    return candidates, elapsed_ms

def rerank_candidates(query, candidates):
    pairs = [(query, item["text"]) for item in candidates]
    sync_cuda()
    start = time.perf_counter()
    scores = reranker.predict(pairs, batch_size=16, show_progress_bar=False, convert_to_numpy=True)
    sync_cuda()
    elapsed_ms = (time.perf_counter() - start) * 1000.0
    reranked = []
    for item, score in zip(candidates, np.asarray(scores).reshape(-1)):
        copied = dict(item)
        copied["rerank_score"] = float(score)
        reranked.append(copied)
    reranked.sort(key=lambda item: item["rerank_score"], reverse=True)
    for rank, item in enumerate(reranked, 1):
        item["rerank_rank"] = int(rank)
    return reranked, elapsed_ms

retrieval_results, ranking_rows, latency_rows = {}, [], []
for qid, query in enumerate(tqdm(ALL_QUERIES, desc="Retrieve + rerank"), 1):
    dense, dense_ms = dense_retrieve(query)
    reranked, rerank_ms = rerank_candidates(query, dense)
    reranked_by_id = {x["chunk_id"]: x for x in reranked}
    spec = query_metadata[query]
    for item in dense:
        rr = reranked_by_id[item["chunk_id"]]
        ranking_rows.append({
            "query_id": qid, "query_type": spec["query_type"],
            "failure_mode_candidate": spec["failure_mode_candidate"], "query": query,
            "chunk_id": item["chunk_id"], "page": item["page"],
            "dense_rank": item["dense_rank"], "dense_score": item["dense_score"],
            "rerank_rank": rr["rerank_rank"], "rerank_score": rr["rerank_score"],
            "rank_shift_positive_means_up": item["dense_rank"] - rr["rerank_rank"],
            "promoted_into_top3": bool(item["dense_rank"] > TOP_K_CONTEXT and rr["rerank_rank"] <= TOP_K_CONTEXT),
            "text_preview": item["text"][:500],
        })
    retrieval_results[query] = {
        "query_id": qid, "query_type": spec["query_type"],
        "failure_mode_candidate": spec["failure_mode_candidate"],
        "dense_top10": dense, "reranked_top10": reranked,
        "baseline_top3": dense[:TOP_K_CONTEXT], "reranked_top3": reranked[:TOP_K_CONTEXT],
        "dense_retrieval_ms": float(dense_ms), "cross_encoder_rerank_ms": float(rerank_ms),
    }
    latency_rows.append({
        "query_id": qid, "query_type": spec["query_type"], "query": query,
        "dense_retrieval_ms": float(dense_ms), "cross_encoder_rerank_ms": float(rerank_ms),
    })
    # Checkpoint progress after every question.
    pd.DataFrame(ranking_rows).to_csv(OUTPUT_DIR / "retrieval_rankings.csv", index=False)
    pd.DataFrame(latency_rows).to_csv(OUTPUT_DIR / "retrieval_latency.csv", index=False)

rankings_df = pd.DataFrame(ranking_rows)
retrieval_latency_df = pd.DataFrame(latency_rows)
expected_rows = len(ALL_QUERIES) * min(TOP_K_CANDIDATES, len(chunks))
assert len(rankings_df) == expected_rows
print("Rank rows:", len(rankings_df), "| expected:", expected_rows)
print("All cross-encoder rankings are restricted to the dense Top 10.")
display(retrieval_latency_df)

Retrieve + rerank:   0%|          | 0/10 [00:00<?, ?it/s]

Rank rows: 100 | expected: 100
All cross-encoder rankings are restricted to the dense Top 10.


,query_id,query_type,query,dense_retrieval_ms,cross_encoder_rerank_ms
0,1,broad coverage,How does Adobe incorporate generative AI into ...,33.961525,407.564900
1,2,stress-test candidate,"What cybersecurity risks does Adobe disclose, ...",14.812491,263.019638
2,3,numeric disclosure,What were Adobe's Digital Media and Digital Ex...,15.128914,262.131684
3,4,broad coverage,What major factors could adversely affect Adob...,13.295950,267.171096
4,5,broad coverage,What changes does Adobe report in its financia...,14.393549,262.762727
5,6,stress-test candidate,"What legal proceedings does Adobe disclose, in...",14.635252,276.289538
6,7,stress-test candidate,What qualifications or limitations apply to Ad...,13.532194,264.992140
7,8,numeric disclosure,What were subscription revenues from Creative ...,13.196890,263.699961
8,9,stress-test candidate,"Which cybersecurity oversight, risk assessment...",14.486418,263.487637
9,10,stress-test candidate,What does Adobe disclose about indemnifying cu...,15.612278,265.139324


## 7. Export rank shifts, Top-3 overlap, and full-text review candidates

A positive rank shift means a chunk moved upward. `candidate_failure_cases.csv` contains each of the ten retrieved candidates per query, not only promoted chunks, and includes full passage text and manual-review fields.

In [8]:
rank_shift_summary = (
    rankings_df.assign(
        moved_up=rankings_df["rank_shift_positive_means_up"] > 0,
        moved_down=rankings_df["rank_shift_positive_means_up"] < 0,
    )
    .groupby(["query_id", "query_type", "failure_mode_candidate", "query"], as_index=False)
    .agg(
        candidates_scored=("chunk_id", "count"),
        candidates_moved_up=("moved_up", "sum"),
        candidates_moved_down=("moved_down", "sum"),
        max_upward_shift=("rank_shift_positive_means_up", "max"),
        promoted_into_top3=("promoted_into_top3", "sum"),
    )
)
rank_shift_summary.to_csv(OUTPUT_DIR / "rank_shift_summary.csv", index=False)

overlap_rows = []
for qid, query in enumerate(ALL_QUERIES, 1):
    rr = retrieval_results[query]
    a = {x["chunk_id"] for x in rr["baseline_top3"]}
    b = {x["chunk_id"] for x in rr["reranked_top3"]}
    overlap_rows.append({
        "query_id": qid, "query_type": rr["query_type"], "query": query,
        "shared_top3_chunks": len(a & b),
        "changed_top3_chunks": len(a ^ b) // 2,
        "jaccard_top3_overlap": len(a & b) / max(1, len(a | b)),
    })
overlap_df = pd.DataFrame(overlap_rows)
overlap_df.to_csv(OUTPUT_DIR / "top3_overlap.csv", index=False)

chunk_lookup = {c["chunk_id"]: c for c in chunks}
candidate_rows = []
for row in rankings_df.to_dict(orient="records"):
    candidate_rows.append({
        **row,
        "full_text": chunk_lookup[int(row["chunk_id"])]["text"],
        "human_relevant": "",
        "baseline_top3_lacks_this_evidence": "",
        "review_notes": "",
    })
candidate_review_df = pd.DataFrame(candidate_rows)
candidate_review_df.to_csv(OUTPUT_DIR / "candidate_failure_cases.csv", index=False)

print("Promoted candidates:", int(rankings_df["promoted_into_top3"].sum()))
print("Promotion alone is not proof of success; review full text and paired answers.")
display(rank_shift_summary)
display(overlap_df)

Promoted candidates: 10
Promotion alone is not proof of success; review full text and paired answers.


,query_id,query_type,failure_mode_candidate,query,candidates_scored,candidates_moved_up,candidates_moved_down,max_upward_shift,promoted_into_top3
0,1,broad coverage,Terminology / risk-context distractor,How does Adobe incorporate generative AI into ...,10,5,4,5,1
1,2,stress-test candidate,Keyword trap / governance details,"What cybersecurity risks does Adobe disclose, ...",10,6,4,6,0
2,3,numeric disclosure,Similar segment names / numeric distractor,What were Adobe's Digital Media and Digital Ex...,10,4,4,4,1
3,4,broad coverage,General risk disclosure,What major factors could adversely affect Adob...,10,5,3,4,2
4,5,broad coverage,Financial results / similar figures,What changes does Adobe report in its financia...,10,3,6,6,1
5,6,stress-test candidate,Specific legal detail vs general litigation,"What legal proceedings does Adobe disclose, in...",10,3,5,5,1
6,7,stress-test candidate,Exception / limitation,What qualifications or limitations apply to Ad...,10,4,3,4,1
7,8,numeric disclosure,Similar customer-group labels / numeric distra...,What were subscription revenues from Creative ...,10,5,4,5,1
8,9,stress-test candidate,Keyword trap / distinct terminology,"Which cybersecurity oversight, risk assessment...",10,6,3,4,1
9,10,stress-test candidate,Exception / role-specific constraint,What does Adobe disclose about indemnifying cu...,10,4,4,6,1


,query_id,query_type,query,shared_top3_chunks,changed_top3_chunks,jaccard_top3_overlap
0,1,broad coverage,How does Adobe incorporate generative AI into ...,2,1,0.5
1,2,stress-test candidate,"What cybersecurity risks does Adobe disclose, ...",3,0,1.0
2,3,numeric disclosure,What were Adobe's Digital Media and Digital Ex...,2,1,0.5
3,4,broad coverage,What major factors could adversely affect Adob...,1,2,0.2
4,5,broad coverage,What changes does Adobe report in its financia...,2,1,0.5
5,6,stress-test candidate,"What legal proceedings does Adobe disclose, in...",2,1,0.5
6,7,stress-test candidate,What qualifications or limitations apply to Ad...,2,1,0.5
7,8,numeric disclosure,What were subscription revenues from Creative ...,2,1,0.5
8,9,stress-test candidate,"Which cybersecurity oversight, risk assessment...",2,1,0.5
9,10,stress-test candidate,What does Adobe disclose about indemnifying cu...,2,1,0.5


## 8. Release retrieval models and load a tested local generator

After retrieving and reranking, embeddings and results are saved, so the encoder models are released to free VRAM. The generator loader tries a public 3B Qwen instruct model and then 1.5B. On GPU it prefers 4-bit weights where possible. Each candidate must pass a finite-logit forward test and produce a nonempty sample answer before use.

In [9]:
del reranker
del embedder
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

from transformers import AutoModelForCausalLM, AutoTokenizer
try:
    from transformers import BitsAndBytesConfig
except Exception:
    BitsAndBytesConfig = None

def chat_inputs(tok, model, system_text, user_text):
    messages = [
        {"role": "system", "content": system_text},
        {"role": "user", "content": user_text},
    ]
    encoded = tok.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True,
        return_dict=True, return_tensors="pt",
    )
    device = model.get_input_embeddings().weight.device
    return {key: value.to(device) for key, value in encoded.items()}

def load_and_test_generator(model_id):
    start = time.perf_counter()
    tok = AutoTokenizer.from_pretrained(model_id, use_fast=True)
    if tok.pad_token_id is None:
        tok.pad_token = tok.eos_token
    tok.padding_side = "left"
    model = None
    load_notes = []

    if torch.cuda.is_available() and USE_4BIT_GENERATOR_WHEN_GPU and BitsAndBytesConfig is not None:
        try:
            compute_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
            qconfig = BitsAndBytesConfig(
                load_in_4bit=True, bnb_4bit_quant_type="nf4",
                bnb_4bit_compute_dtype=compute_dtype, bnb_4bit_use_double_quant=True,
            )
            model = AutoModelForCausalLM.from_pretrained(
                model_id, device_map="auto", quantization_config=qconfig,
                torch_dtype=compute_dtype, low_cpu_mem_usage=True,
            ).eval()
        except Exception as exc:
            load_notes.append(f"4-bit load failed: {type(exc).__name__}: {exc}")
            if model is not None:
                del model
                model = None
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

    if model is None:
        if torch.cuda.is_available():
            dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
            model = AutoModelForCausalLM.from_pretrained(
                model_id, torch_dtype=dtype, device_map="auto", low_cpu_mem_usage=True,
            ).eval()
        else:
            model = AutoModelForCausalLM.from_pretrained(
                model_id, torch_dtype=torch.float32, low_cpu_mem_usage=True,
            ).eval()

    test_system = "You are a concise assistant."
    test_user = "Reply with one short sentence: What is Adobe?"
    test_inputs = chat_inputs(tok, model, test_system, test_user)
    with torch.inference_mode():
        test_outputs = model(**test_inputs, use_cache=False)
    if not torch.isfinite(test_outputs.logits).all().item():
        raise RuntimeError(f"{model_id} returned NaN/Inf logits on the test prompt.")
    input_length = test_inputs["input_ids"].shape[-1]
    with torch.inference_mode():
        generated = model.generate(
            **test_inputs, max_new_tokens=32, do_sample=False,
            pad_token_id=tok.pad_token_id, eos_token_id=tok.eos_token_id, use_cache=True,
        )
    answer = tok.decode(generated[0, input_length:], skip_special_tokens=True).strip()
    if not answer or not any(ch.isalpha() for ch in answer):
        raise RuntimeError(f"{model_id} generated an empty/invalid test response.")
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    return tok, model, time.perf_counter() - start, answer, load_notes

tokenizer, generator_model = None, None
GENERATION_MODEL_NAME, generator_load_s, generator_test_answer = None, None, ""
generator_load_errors, generator_status = [], "unavailable"

for model_id in GENERATION_MODEL_CANDIDATES:
    print("Trying:", model_id)
    try:
        tok, model, elapsed, test_answer, notes = load_and_test_generator(model_id)
        tokenizer, generator_model = tok, model
        GENERATION_MODEL_NAME, generator_load_s = model_id, elapsed
        generator_test_answer, generator_load_errors = test_answer, generator_load_errors + notes
        generator_status = "ready"
        print("Self-test passed:", test_answer)
        break
    except Exception as exc:
        error = f"{model_id}: {type(exc).__name__}: {exc}"
        generator_load_errors.append(error)
        print("Candidate rejected:", error[:600])
        try:
            if "model" in locals() and model is not None:
                del model
        except Exception:
            pass
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

if generator_status != "ready":
    print("WARNING: No generator passed validation. Retrieval artifacts will still be zipped.")
    print("\n".join(generator_load_errors[-4:]))

pd.DataFrame([{
    "component": "generation_model_load_and_self_test",
    "model": GENERATION_MODEL_NAME or "none",
    "seconds": generator_load_s,
    "status": generator_status,
    "test_answer": generator_test_answer,
    "load_errors": " || ".join(generator_load_errors),
}]).to_csv(OUTPUT_DIR / "generation_model_profile.csv", index=False)

print("Chosen generator:", GENERATION_MODEL_NAME or "none")
print("Generator status:", generator_status)

Trying: Qwen/Qwen2.5-3B-Instruct


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

model-00001-of-00002.safetensors:   0%|          | 0.00/3.97G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Self-test passed: Adobe is a company that provides software solutions including Photoshop, Illustrator, and Acrobat Pro.
Chosen generator: Qwen/Qwen2.5-3B-Instruct
Generator status: ready


## 9. Generate paired answers using identical prompts/settings

The system prompt and user prompt structure stay the same for each query. Only the selected context changes: dense Top 3 for the baseline and reranked Top 3 for the advanced pipeline. The notebook saves a checkpoint after each answer and records errors explicitly.

In [10]:
# SECTION 9 — Generate paired answers with completion checks

GENERATION_MAX_NEW_TOKENS = 1024

SYSTEM_INSTRUCTION = (
    "You are a careful assistant answering questions about Adobe Inc.'s "
    "fiscal 2025 Form 10-K. Use only the supplied context excerpts. "
    "Do not invent facts, numbers, or legal conclusions. Cite source PDF "
    "page numbers in parentheses, for example (p. 42). Distinguish "
    "reported facts from risks or projections. Answer every part of the "
    "question. For questions about multiple metrics, years, or categories, "
    "address each one explicitly. Preserve the exact metric being asked "
    "about; do not substitute total revenue for subscription revenue or "
    "one category for another. If the supplied context does not establish "
    "a requested fact, say so clearly. Finish every sentence and list item."
)

PROMPT_VERSION = "same_system_user_chat_template_v2_1024_tokens"


def format_context(items):
    return "\n\n---\n\n".join(
        f"[Source: PDF page {x['page']}; chunk {x['chunk_id']}]\n{x['text']}"
        for x in items
    )


def build_user_prompt(query, context_chunks):
    return (
        f"Question: {query}\n\n"
        "Retrieved context excerpts from Adobe's fiscal 2025 Form 10-K:\n"
        f"{format_context(context_chunks)}\n\n"
        "Instructions:\n"
        "1. Answer the exact question using only the excerpts.\n"
        "2. Address every requested part, including all years and categories.\n"
        "3. For numerical comparisons, state the metric, period, value, "
        "and percentage change when available.\n"
        "4. Cite relevant PDF pages for important claims.\n"
        "5. If a requested value is not established by the excerpts, "
        "say so instead of guessing.\n"
        "6. End with a complete sentence."
    )


def generate_answer(query, context_chunks):
    if generator_model is None or tokenizer is None:
        return (
            "",
            None,
            "failed_model_unavailable",
            "No generator passed startup validation.",
        )

    try:
        prompt = build_user_prompt(query, context_chunks)
        encoded = chat_inputs(
            tokenizer,
            generator_model,
            SYSTEM_INSTRUCTION,
            prompt,
        )
        input_length = encoded["input_ids"].shape[-1]

        if torch.cuda.is_available():
            torch.cuda.synchronize()

        start = time.perf_counter()

        with torch.inference_mode():
            output = generator_model.generate(
                **encoded,
                max_new_tokens=GENERATION_MAX_NEW_TOKENS,
                do_sample=False,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id,
                use_cache=True,
            )

        if torch.cuda.is_available():
            torch.cuda.synchronize()

        elapsed_ms = (time.perf_counter() - start) * 1000.0

        new_tokens = output[0, input_length:]
        answer = tokenizer.decode(
            new_tokens,
            skip_special_tokens=True,
        ).strip()

        if not answer or not any(ch.isalpha() for ch in answer):
            return (
                answer,
                elapsed_ms,
                "failed_empty_output",
                "Decoded output was empty or invalid.",
            )

        # Reaching the full generation allowance may indicate truncation.
        if len(new_tokens) >= GENERATION_MAX_NEW_TOKENS:
            return (
                answer,
                elapsed_ms,
                "check_possible_truncation",
                f"Generated {len(new_tokens)} tokens; inspect the answer ending.",
            )

        return answer, elapsed_ms, "success", ""

    except Exception as exc:
        return (
            "",
            None,
            "failed_exception",
            f"{type(exc).__name__}: {exc}",
        )


# Regenerate both pipelines using the same model and generation settings.
generation_rows = []
generation_path = OUTPUT_DIR / "generation_comparison_long.csv"

for qid, query in enumerate(
    tqdm(ALL_QUERIES, desc="Regenerate both answers"),
    1,
):
    result = retrieval_results[query]

    pipelines = [
        ("bi_encoder_top3", result["baseline_top3"]),
        (
            "cross_encoder_reranked_top3",
            result["reranked_top3"],
        ),
    ]

    for pipeline_name, context_chunks in pipelines:
        answer, gen_ms, status, error = generate_answer(
            query,
            context_chunks,
        )

        row = {
            "query_id": qid,
            "query_type": result["query_type"],
            "failure_mode_candidate": result["failure_mode_candidate"],
            "query": query,
            "pipeline": pipeline_name,
            "context_chunk_ids": json.dumps(
                [x["chunk_id"] for x in context_chunks]
            ),
            "context_pages": json.dumps(
                [x["page"] for x in context_chunks]
            ),
            "answer": answer,
            "generation_ms": gen_ms,
            "status": status,
            "error": error,
            "generation_model": GENERATION_MODEL_NAME or "none",
            "prompt_version": PROMPT_VERSION,
        }

        generation_rows.append(row)

        # Save progress after every answer.
        pd.DataFrame(generation_rows).to_csv(
            generation_path,
            index=False,
        )

        print(
            f"Q{qid:02d} | {pipeline_name} | {status} | "
            f"{gen_ms if gen_ms is not None else 'n/a'} ms"
        )

generation_df = pd.DataFrame(generation_rows)

print("\nAnswer generation status:")
print(generation_df["status"].value_counts(dropna=False))

answer_nonempty = (
    generation_df["answer"]
    .fillna("")
    .astype(str)
    .str.strip()
    .ne("")
)

print(
    "\nNon-empty answers:",
    int(answer_nonempty.sum()),
    "/",
    len(generation_df),
)

display(
    generation_df[
        ["query_id", "pipeline", "status", "answer", "error"]
    ]
)

Regenerate both answers:   0%|          | 0/10 [00:00<?, ?it/s]

Q01 | bi_encoder_top3 | success | 24929.653185000006 ms
Q01 | cross_encoder_reranked_top3 | success | 27844.017109999982 ms
Q02 | bi_encoder_top3 | success | 44861.80298200003 ms
Q02 | cross_encoder_reranked_top3 | success | 42213.141938999965 ms
Q03 | bi_encoder_top3 | success | 26641.084602000035 ms
Q03 | cross_encoder_reranked_top3 | success | 29829.647487000046 ms
Q04 | bi_encoder_top3 | success | 33816.474482999976 ms
Q04 | cross_encoder_reranked_top3 | success | 28396.164999999997 ms
Q05 | bi_encoder_top3 | success | 62038.20769499998 ms
Q05 | cross_encoder_reranked_top3 | success | 52677.68358399996 ms
Q06 | bi_encoder_top3 | success | 75935.13495800005 ms
Q06 | cross_encoder_reranked_top3 | success | 31931.30941300001 ms
Q07 | bi_encoder_top3 | success | 45862.60710400006 ms
Q07 | cross_encoder_reranked_top3 | success | 53723.99703200006 ms
Q08 | bi_encoder_top3 | success | 34863.89623900004 ms
Q08 | cross_encoder_reranked_top3 | success | 28093.29431499998 ms
Q09 | bi_encoder_

,query_id,pipeline,status,answer,error
0,1,bi_encoder_top3,success,Adobe incorporates generative AI into its prod...,
1,1,cross_encoder_reranked_top3,success,Adobe incorporates generative AI into its prod...,
2,2,bi_encoder_top3,success,Adobe discloses cybersecurity risks related to...,
3,2,cross_encoder_reranked_top3,success,Adobe discloses cybersecurity risks related to...,
4,3,bi_encoder_top3,success,Adobe's Digital Media and Digital Experience s...,
5,3,cross_encoder_reranked_top3,success,Adobe's Digital Media and Digital Experience s...,
6,4,bi_encoder_top3,success,The major factors that could adversely affect ...,
7,4,cross_encoder_reranked_top3,success,The major factors that could adversely affect ...,
8,5,bi_encoder_top3,success,Adobe reports significant changes in its finan...,
9,5,cross_encoder_reranked_top3,success,Adobe reports significant growth in its financ...,


## 10. Export side-by-side answer and context comparisons

In [11]:
def record_lookup():
    return {(r["query_id"], r["pipeline"]): r for r in generation_rows}

answer_records = record_lookup()
comparison_rows, context_rows = [], []
for qid, query in enumerate(ALL_QUERIES, 1):
    result = retrieval_results[query]
    base = answer_records.get((qid, "bi_encoder_top3"), {})
    rerank = answer_records.get((qid, "cross_encoder_reranked_top3"), {})
    comparison_rows.append({
        "query_id": qid, "query_type": result["query_type"],
        "failure_mode_candidate": result["failure_mode_candidate"], "query": query,
        "baseline_answer": base.get("answer", ""), "baseline_status": base.get("status", "missing"),
        "baseline_generation_ms": base.get("generation_ms"),
        "reranked_answer": rerank.get("answer", ""), "reranked_status": rerank.get("status", "missing"),
        "reranked_generation_ms": rerank.get("generation_ms"),
        "baseline_context_chunk_ids": json.dumps([x["chunk_id"] for x in result["baseline_top3"]]),
        "reranked_context_chunk_ids": json.dumps([x["chunk_id"] for x in result["reranked_top3"]]),
    })
    context_rows.append({
        "query_id": qid, "query_type": result["query_type"],
        "failure_mode_candidate": result["failure_mode_candidate"], "query": query,
        "baseline_top3_context": format_context(result["baseline_top3"]),
        "reranked_top3_context": format_context(result["reranked_top3"]),
    })

generation_comparison_df = pd.DataFrame(comparison_rows)
context_comparison_df = pd.DataFrame(context_rows)
generation_comparison_df.to_csv(OUTPUT_DIR / "generation_comparison.csv", index=False)
context_comparison_df.to_csv(OUTPUT_DIR / "top3_context_comparison.csv", index=False)
display(generation_comparison_df[[
    "query_id", "query", "baseline_status", "reranked_status",
    "baseline_generation_ms", "reranked_generation_ms",
]])

,query_id,query,baseline_status,reranked_status,baseline_generation_ms,reranked_generation_ms
0,1,How does Adobe incorporate generative AI into ...,success,success,24929.653185,27844.017110
1,2,"What cybersecurity risks does Adobe disclose, ...",success,success,44861.802982,42213.141939
2,3,What were Adobe's Digital Media and Digital Ex...,success,success,26641.084602,29829.647487
3,4,What major factors could adversely affect Adob...,success,success,33816.474483,28396.165000
4,5,What changes does Adobe report in its financia...,success,success,62038.207695,52677.683584
5,6,"What legal proceedings does Adobe disclose, in...",success,success,75935.134958,31931.309413
6,7,What qualifications or limitations apply to Ad...,success,success,45862.607104,53723.997032
7,8,What were subscription revenues from Creative ...,success,success,34863.896239,28093.294315
8,9,"Which cybersecurity oversight, risk assessment...",success,success,26015.780714,32085.872847
9,10,What does Adobe disclose about indemnifying cu...,success,success,54611.676116,33734.701938


## 11. Latency profiling and integrity checks

Baseline end-to-end time is dense retrieval plus its generation call. Reranked end-to-end time is dense retrieval plus cross-encoder scoring plus its generation call. Startup/model-embedding costs are measured separately.

In [12]:
# SECTION 11 — Latency profiling and integrity checks

generation_latency_pivot = generation_df.pivot_table(
    index="query_id",
    columns="pipeline",
    values="generation_ms",
    aggfunc="first",
).reset_index()

generation_status_pivot = generation_df.pivot_table(
    index="query_id",
    columns="pipeline",
    values="status",
    aggfunc="first",
).reset_index()

latency_summary = retrieval_latency_df.merge(
    generation_latency_pivot,
    on="query_id",
    how="left",
)

latency_summary = latency_summary.merge(
    generation_status_pivot,
    on="query_id",
    how="left",
    suffixes=("", "_status"),
)

latency_summary = latency_summary.rename(
    columns={
        "bi_encoder_top3": "baseline_generation_ms",
        "cross_encoder_reranked_top3": "reranked_generation_ms",
        "bi_encoder_top3_status": "baseline_generation_status",
        "cross_encoder_reranked_top3_status": "reranked_generation_status",
    }
)

latency_summary["baseline_end_to_end_ms"] = (
    latency_summary["dense_retrieval_ms"]
    + latency_summary["baseline_generation_ms"]
)

latency_summary["reranked_end_to_end_ms"] = (
    latency_summary["dense_retrieval_ms"]
    + latency_summary["cross_encoder_rerank_ms"]
    + latency_summary["reranked_generation_ms"]
)

latency_summary.to_csv(
    OUTPUT_DIR / "latency_summary.csv",
    index=False,
)

answer_nonempty = (
    generation_df["answer"]
    .fillna("")
    .astype(str)
    .str.strip()
    .ne("")
)

possible_truncation = (
    generation_df["status"] == "check_possible_truncation"
)

failed_answers = ~generation_df["status"].isin(
    ["success", "check_possible_truncation"]
)

expected_answers = len(ALL_QUERIES) * 2
expected_rankings = len(ALL_QUERIES) * min(
    TOP_K_CANDIDATES,
    len(chunks),
)

integrity = {
    "corpus_source": corpus_source,
    "pages_or_text_records_loaded": int(len(pages)),
    "chunks": int(len(chunks)),
    "queries_evaluated": int(len(ALL_QUERIES)),
    "expected_ranking_rows": int(expected_rankings),
    "actual_ranking_rows": int(len(rankings_df)),
    "expected_answers": int(expected_answers),
    "answer_rows": int(len(generation_df)),
    "answers_nonempty": int(answer_nonempty.sum()),
    "answers_status_success": int(
        (generation_df["status"] == "success").sum()
    ),
    "answers_possible_truncation": int(possible_truncation.sum()),
    "answers_failed": int(failed_answers.sum()),
    "generator_status": generator_status,
    "generation_model": GENERATION_MODEL_NAME or "none",
    "all_answers_nonempty": bool(
        len(generation_df) == expected_answers
        and answer_nonempty.all()
    ),
    "all_answers_passed_generation_checks": bool(
        len(generation_df) == expected_answers
        and (generation_df["status"] == "success").all()
        and answer_nonempty.all()
    ),
    "promoted_candidates_for_human_review": int(
        rankings_df["promoted_into_top3"].sum()
    ),
    "note": (
        "Successful generation does not prove factual correctness or "
        "completeness. Inspect answers manually. Rank promotion alone "
        "does not establish genuine reranking improvement."
    ),
}

(OUTPUT_DIR / "integrity_check.json").write_text(
    json.dumps(integrity, indent=2),
    encoding="utf-8",
)

print(json.dumps(integrity, indent=2))

print("\nMean latency in milliseconds:")
display(
    latency_summary[
        [
            "dense_retrieval_ms",
            "cross_encoder_rerank_ms",
            "baseline_generation_ms",
            "reranked_generation_ms",
            "baseline_end_to_end_ms",
            "reranked_end_to_end_ms",
        ]
    ].mean(numeric_only=True).to_frame("mean_ms")
)

print("\nAnswers needing manual inspection:")
display(
    generation_df.loc[
        possible_truncation | failed_answers,
        ["query_id", "pipeline", "status", "error", "answer"],
    ]
)

{
  "corpus_source": "/kaggle/input/datasets/vidurshah04/adobe-extracted",
  "pages_or_text_records_loaded": 93,
  "chunks": 207,
  "queries_evaluated": 10,
  "expected_ranking_rows": 100,
  "actual_ranking_rows": 100,
  "expected_answers": 20,
  "answer_rows": 20,
  "answers_nonempty": 20,
  "answers_status_success": 20,
  "answers_possible_truncation": 0,
  "answers_failed": 0,
  "generator_status": "ready",
  "generation_model": "Qwen/Qwen2.5-3B-Instruct",
  "all_answers_nonempty": true,
  "all_answers_passed_generation_checks": true,
  "promoted_candidates_for_human_review": 10,
  "note": "Successful generation does not prove factual correctness or completeness. Inspect answers manually. Rank promotion alone does not establish genuine reranking improvement."
}

Mean latency in milliseconds:


,mean_ms
dense_retrieval_ms,16.305546
cross_encoder_rerank_ms,279.625865
baseline_generation_ms,42957.631808
reranked_generation_ms,36052.983066
baseline_end_to_end_ms,42973.937354
reranked_end_to_end_ms,36348.914477



Answers needing manual inspection:


,query_id,pipeline,status,error,answer


## 12. Create the human failure-case review template

The two review files help identify at least two real cases. `candidate_failure_cases.csv` contains individual full-text candidates and rank shifts; `failure_case_review_template.csv` summarizes both Top-3 contexts and answers for each query. Fill in the manual-review columns before writing report conclusions.

In [13]:
review_rows = []
for qid, query in enumerate(ALL_QUERIES, 1):
    result = retrieval_results[query]
    base = answer_records.get((qid, "bi_encoder_top3"), {})
    rerank = answer_records.get((qid, "cross_encoder_reranked_top3"), {})
    review_rows.append({
        "query_id": qid, "query": query,
        "failure_mode_candidate": result["failure_mode_candidate"],
        "baseline_top3_chunk_ids": json.dumps([x["chunk_id"] for x in result["baseline_top3"]]),
        "reranked_top3_chunk_ids": json.dumps([x["chunk_id"] for x in result["reranked_top3"]]),
        "baseline_top3_full_text": format_context(result["baseline_top3"]),
        "reranked_top3_full_text": format_context(result["reranked_top3"]),
        "baseline_answer": base.get("answer", ""),
        "reranked_answer": rerank.get("answer", ""),
        "is_genuine_failure": "",
        "what_baseline_missed": "",
        "why_reranked_context_is_better": "",
        "answer_difference_notes": "",
    })
failure_review_df = pd.DataFrame(review_rows)
failure_review_df.to_csv(OUTPUT_DIR / "failure_case_review_template.csv", index=False)
print("Review template created. No candidate has been automatically marked as a genuine success.")

Review template created. No candidate has been automatically marked as a genuine success.


## 13. Save the complete JSON record and environment metadata

In [14]:
def json_safe(value):
    if value is None:
        return None
    if isinstance(value, dict):
        return {str(k): json_safe(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_safe(v) for v in value]
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.floating,)):
        return float(value) if np.isfinite(value) else None
    if isinstance(value, (np.bool_,)):
        return bool(value)
    if isinstance(value, float) and not np.isfinite(value):
        return None
    if isinstance(value, (str, int, bool)):
        return value
    try:
        if pd.isna(value):
            return None
    except (TypeError, ValueError):
        pass
    return str(value)

full_payload = {
    "metadata": {
        "created_at": time.strftime("%Y-%m-%d %H:%M:%S"),
        "corpus_source": corpus_source,
        "pages_or_text_records_loaded": len(pages),
        "num_chunks": len(chunks),
        "embedding_model": EMBEDDING_MODEL_NAME,
        "reranker_model": RERANKER_MODEL_NAME,
        "generation_model": GENERATION_MODEL_NAME or "none",
        "generator_status": generator_status,
        "prompt_version": PROMPT_VERSION,
        "top_k_candidates": TOP_K_CANDIDATES,
        "top_k_context": TOP_K_CONTEXT,
        "chunk_size_words": CHUNK_SIZE_WORDS,
        "chunk_overlap_words": CHUNK_OVERLAP_WORDS,
        "embedding_matrix_shape": list(embeddings.shape),
    },
    "integrity": integrity,
    "rank_shift_summary": rank_shift_summary.to_dict(orient="records"),
    "top3_overlap": overlap_df.to_dict(orient="records"),
    "latency_summary": latency_summary.to_dict(orient="records"),
    "generation_results": generation_rows,
    "queries": [],
    "generation_max_new_tokens": GENERATION_MAX_NEW_TOKENS,
}
for query in ALL_QUERIES:
    result = retrieval_results[query]
    full_payload["queries"].append({
        "query_id": result["query_id"], "query_type": result["query_type"],
        "failure_mode_candidate": result["failure_mode_candidate"], "query": query,
        "dense_retrieval_ms": result["dense_retrieval_ms"],
        "cross_encoder_rerank_ms": result["cross_encoder_rerank_ms"],
        "dense_top10": result["dense_top10"],
        "reranked_top10": result["reranked_top10"],
        "baseline_top3": result["baseline_top3"],
        "reranked_top3": result["reranked_top3"],
    })

(OUTPUT_DIR / "full_pipeline_results.json").write_text(
    json.dumps(json_safe(full_payload), ensure_ascii=False, indent=2, allow_nan=False),
    encoding="utf-8",
)
environment = {
    "python": platform.python_version(),
    "torch": torch.__version__,
    "cuda_available": bool(torch.cuda.is_available()),
    "gpu_names": [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]
        if torch.cuda.is_available() else [],
    "transformers": __import__("transformers").__version__,
    "sentence_transformers": __import__("sentence_transformers").__version__,
}
(OUTPUT_DIR / "environment.json").write_text(json.dumps(environment, indent=2), encoding="utf-8")
print("Saved full_pipeline_results.json and environment.json")

Saved full_pipeline_results.json and environment.json


## 14. Zip all output artifacts

The ZIP contains every file in `rag_outputs/`: normalized chunks, embeddings, Top-10 ranks/scores, answer comparisons, full contexts, latency tables, failure-review templates, environment metadata, and full JSON results. The ZIP is saved outside the output directory so it does not include itself.

In [15]:
if ZIP_PATH.exists():
    ZIP_PATH.unlink()
with zipfile.ZipFile(ZIP_PATH, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=6) as archive:
    for path in sorted(OUTPUT_DIR.rglob("*")):
        if path.is_file():
            archive.write(path, arcname=str(Path("rag_outputs") / path.relative_to(OUTPUT_DIR)))

print("ZIP created:", ZIP_PATH.resolve())
print("ZIP size (MiB):", round(ZIP_PATH.stat().st_size / (1024 ** 2), 2))
print("\nFiles included:")
for path in sorted(OUTPUT_DIR.rglob("*")):
    if path.is_file():
        print(f"{path.relative_to(OUTPUT_DIR)} — {path.stat().st_size / 1024:.1f} KiB")
print("\nFinal integrity status:")
print(json.dumps(integrity, indent=2))
print("\nBefore report writing, manually verify at least two genuine baseline failure cases.")

ZIP created: /kaggle/working/rag_assignment_outputs.zip
ZIP size (MiB): 0.91

Files included:
candidate_failure_cases.csv — 281.6 KiB
corpus_source.txt — 0.1 KiB
embeddings.npy — 621.1 KiB
environment.json — 0.2 KiB
evaluation_queries.csv — 1.9 KiB
failure_case_review_template.csv — 165.0 KiB
full_pipeline_results.json — 660.0 KiB
generation_comparison.csv — 43.8 KiB
generation_comparison_long.csv — 47.7 KiB
generation_model_profile.csv — 0.2 KiB
integrity_check.json — 0.8 KiB
latency_summary.csv — 2.9 KiB
model_load_profile.csv — 0.1 KiB
normalized_chunks.json — 398.2 KiB
rank_shift_summary.csv — 2.1 KiB
retrieval_latency.csv — 1.9 KiB
retrieval_rankings.csv — 73.7 KiB
top3_context_comparison.csv — 123.8 KiB
top3_overlap.csv — 1.6 KiB

Final integrity status:
{
  "corpus_source": "/kaggle/input/datasets/vidurshah04/adobe-extracted",
  "pages_or_text_records_loaded": 93,
  "chunks": 207,
  "queries_evaluated": 10,
  "expected_ranking_rows": 100,
  "actual_ranking_rows": 100,
  "expecte